In [1]:
# ==========================================================
# Import Libraries
# ==========================================================

from pyspark.sql import functions as F

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 3, Finished, Available, Finished, False)

In [2]:
# ==========================================================
# Read Silver Tables
# ==========================================================

customers = spark.table("silver_customers")

products = spark.table("silver_products")

stores = spark.table("silver_stores")

employees = spark.table("silver_employees")

sales = spark.table("silver_sales")

returns = spark.table("silver_returns")

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 4, Finished, Available, Finished, False)

In [3]:
# ==========================================================
# Customer Dimension
# ==========================================================

dim_customer = (

    customers

    .select(

        "CustomerID",

        "FirstName",

        "LastName",

        "Gender",

        "BirthDate",

        "City",

        "Region",

        "Country",

        "LoyaltyLevel",

        "JoinDate"

    )

)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 5, Finished, Available, Finished, False)

In [4]:
(
    dim_customer.write

    .mode("overwrite")

    .format("delta")

    .saveAsTable("dim_customer")
)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 6, Finished, Available, Finished, False)

In [5]:
# ==========================================================
# Product Dimension
# ==========================================================

dim_product = (

    products

    .select(

        "ProductID",

        "ProductName",

        "Category",

        "SubCategory",

        "Brand",

        "Supplier"

    )

)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 7, Finished, Available, Finished, False)

In [6]:
(
    dim_product.write

    .mode("overwrite")

    .format("delta")

    .saveAsTable("dim_product")
)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 8, Finished, Available, Finished, False)

In [7]:
# ==========================================================
# Store Dimension
# ==========================================================

dim_store = (

    stores

    .select(

        "StoreID",

        "StoreName",

        "City",

        "Region",

        "StoreType"

    )

)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 9, Finished, Available, Finished, False)

In [8]:
(
    dim_store.write

    .mode("overwrite")

    .format("delta")

    .saveAsTable("dim_store")
)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 10, Finished, Available, Finished, False)

In [9]:
# ==========================================================
# Employee Dimension
# ==========================================================

dim_employee = (

    employees

    .select(

        "EmployeeID",

        "EmployeeName",

        "Department",

        "Position",

        "StoreID"

    )

)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 11, Finished, Available, Finished, False)

In [10]:
(
    dim_employee.write

    .mode("overwrite")

    .format("delta")

    .saveAsTable("dim_employee")
)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 12, Finished, Available, Finished, False)

In [11]:
# ==========================================================
# Create Date Dimension
# ==========================================================

from pyspark.sql.functions import *

start_date = "2022-01-01"
end_date = "2025-12-31"

dim_date = (

    spark.sql(f"""

    SELECT explode(

        sequence(

            to_date('{start_date}'),

            to_date('{end_date}'),

            interval 1 day

        )

    ) AS Date

    """)

)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 13, Finished, Available, Finished, False)

In [12]:
# ==========================================================
# Add Date Attributes
# ==========================================================

dim_date = (

    dim_date

    .withColumn(
        "DateKey",
        date_format("Date", "yyyyMMdd").cast("int")
    )

    .withColumn(
        "Year",
        year("Date")
    )

    .withColumn(
        "Quarter",
        quarter("Date")
    )

    .withColumn(
        "Month",
        month("Date")
    )

    .withColumn(
        "MonthName",
        date_format("Date", "MMMM")
    )

    .withColumn(
        "Week",
        weekofyear("Date")
    )

    .withColumn(
        "Day",
        dayofmonth("Date")
    )

    .withColumn(
        "DayName",
        date_format("Date", "EEEE")
    )

    .withColumn(
        "IsWeekend",
        dayofweek("Date").isin([6, 7])
    )

)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 14, Finished, Available, Finished, False)

In [13]:
display(dim_date)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 535169a2-304d-4e50-b64f-ff7d88295813)

In [14]:
(
    dim_date.write

    .mode("overwrite")

    .format("delta")

    .saveAsTable("dim_date")
)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 16, Finished, Available, Finished, False)

In [15]:
fact_sales = (

    sales

    .select(

        "SaleID",

        "CustomerID",

        "ProductID",

        "StoreID",

        "EmployeeID",

        "OrderDate",

        "Quantity",

        "Discount",

        "Revenue",

        "Cost",

        "Profit",

        "Tax",

        "SalesChannel",

        "PaymentMethod",

        "OrderStatus"

    )

)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 17, Finished, Available, Finished, False)

In [16]:
# ==========================================================
# Add DateKey
# ==========================================================

fact_sales = (

    fact_sales

    .withColumn(

        "DateKey",

        date_format(

            "OrderDate",

            "yyyyMMdd"

        ).cast("int")

    )

)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 18, Finished, Available, Finished, False)

In [17]:
display(fact_sales.limit(20))

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cd96e555-062a-4e75-a3fb-b4711c2bdfe2)

In [18]:
fact_sales.printSchema()

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 20, Finished, Available, Finished, False)

root
 |-- SaleID: integer (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- StoreID: integer (nullable = true)
 |-- EmployeeID: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Cost: integer (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Tax: double (nullable = true)
 |-- SalesChannel: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- OrderStatus: string (nullable = true)
 |-- DateKey: integer (nullable = true)



In [19]:
(
    fact_sales.write

    .mode("overwrite")

    .format("delta")

    .saveAsTable("fact_sales")
)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 21, Finished, Available, Finished, False)

In [20]:
spark.sql("""

SHOW TABLES

""").show(truncate=False)

StatementMeta(, e68ce96a-ceb6-49ec-aaa0-f4e54add7787, 22, Finished, Available, Finished, False)

+--------------------------------------+----------------+-----------+
|namespace                             |tableName       |isTemporary|
+--------------------------------------+----------------+-----------+
|`Retail Analytics`.RetailLakehouse.dbo|bronze_customers|false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_employees|false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_products |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_returns  |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_sales    |false      |
|`Retail Analytics`.RetailLakehouse.dbo|bronze_stores   |false      |
|`Retail Analytics`.RetailLakehouse.dbo|dim_customer    |false      |
|`Retail Analytics`.RetailLakehouse.dbo|dim_date        |false      |
|`Retail Analytics`.RetailLakehouse.dbo|dim_employee    |false      |
|`Retail Analytics`.RetailLakehouse.dbo|dim_product     |false      |
|`Retail Analytics`.RetailLakehouse.dbo|dim_store       |false      |
|`Retail Analytics`.